In [1]:
pip install gensim

In [2]:
import pandas as pd
import re
import numpy as np
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tabulate import tabulate

In [3]:
df = pd.read_csv("customer_complaints_1.csv")

In [4]:
df.columns = df.columns.str.strip().str.lower()

In [5]:
df = df[['text']].dropna()

In [6]:
def preprocess_text(text):
    text = str(text).lower()                      
    text = re.sub(r'[^a-zA-Z\s]', '', text)      
    words = text.split()
    words = [word for word in words if word not in ENGLISH_STOP_WORDS] 
    return words

In [7]:
df['tokens'] = df['text'].apply(preprocess_text)

In [8]:
word2vec_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [9]:
def document_vector(tokens, model):
    valid_words = [word for word in tokens if word in model.wv]
    if len(valid_words) == 0:
        return np.zeros(model.vector_size)
    return np.mean([model.wv[word] for word in valid_words], axis=0)

X = np.array([document_vector(tokens, word2vec_model) for tokens in df['tokens']])

In [10]:
k = 3   
km = KMeans(n_clusters=k, random_state=42)
df['cluster'] = km.fit_predict(X)


C:\Users\user\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [11]:
df['cleaned_text'] = df['tokens'].apply(lambda x: " ".join(x))

print("\n=== FIRST 20 CLUSTERING RESULTS (Word2Vec) ===")
print(tabulate(df[['text', 'cluster']].head(20), headers='keys', tablefmt='grid'))



=== FIRST 20 CLUSTERING RESULTS (Word2Vec) ===
+----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
print("\n=== NUMBER OF COMPLAINTS IN EACH CLUSTER ===")
print(df['cluster'].value_counts())


=== NUMBER OF COMPLAINTS IN EACH CLUSTER ===
cluster
1    12
2     6
0     1
Name: count, dtype: int64
